In [1]:
import os, json, re
from typing import List, Dict, Any, Optional, Literal
from datetime import datetime
from IPython.display import Markdown, display

from pydantic import BaseModel, Field, ValidationError

from agents import Agent, Runner, Trace
from agents.mcp import MCPServerStdio

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# ---------------------------
# MCP servers (yours)
# ---------------------------
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}
sql_params = {
    "command": "mssql-mcp-server",
    "env": {"MSSQL_CONNECTION_STRING": os.getenv("MSSQL_CONNECTION_STRING")},
}
chart_params = {"command": "npx", "args": ["-y", "@antv/mcp-server-chart"]}

fetch_server = MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60)
mssql_server = MCPServerStdio(params=sql_params, client_session_timeout_seconds=60)
chart_server = MCPServerStdio(params=chart_params, client_session_timeout_seconds=60)

In [3]:
# ---------------------------
# Pydantic structured outputs
# ---------------------------
class CandidateTable(BaseModel):
    name: str
    confidence: float = Field(ge=0, le=1)
    why: str

class ColumnInfo(BaseModel):
    col: str
    type: Optional[str] = None

class LikelyJoin(BaseModel):
    left: str
    right: str
    confidence: float = Field(ge=0, le=1)
    why: str

class PiiRisk(BaseModel):
    table: str
    cols: List[str]

class EavHint(BaseModel):
    table: str
    entity_id_cols: List[str] = Field(default_factory=list)
    attribute_cols: List[str] = Field(default_factory=list)   # attribute_id/name/key
    value_cols: List[str] = Field(default_factory=list)       # value, string_value, number_value, date_value...
    timestamp_cols: List[str] = Field(default_factory=list)
    confidence: float = Field(ge=0, le=1)
    why: str

class DBCatalog(BaseModel):
    candidate_fact_tables: List[CandidateTable] = Field(default_factory=list)
    table_columns: Dict[str, List[ColumnInfo]] = Field(default_factory=dict)
    candidate_date_columns: Dict[str, List[str]] = Field(default_factory=dict)
    candidate_measure_columns: Dict[str, List[str]] = Field(default_factory=dict)
    likely_joins: List[LikelyJoin] = Field(default_factory=list)
    eav_hints: List[EavHint] = Field(default_factory=list)
    pii_risk_columns: List[PiiRisk] = Field(default_factory=list)
    notes: List[str] = Field(default_factory=list)

class AnswerTable(BaseModel):
    name: str
    columns: List[str]
    rows: List[List[Any]]

class SQLQuery(BaseModel):
    name: str
    sql: str

class QueryRunResult(BaseModel):
    answer_tables: List[AnswerTable]
    assumptions: List[str] = Field(default_factory=list)
    caveats: List[str] = Field(default_factory=list)
    sql_appendix: List[SQLQuery]

class VerificationResult(BaseModel):
    verdict: Literal["PASS", "FAIL"]
    reasons: List[str] = Field(default_factory=list)
    fixes: List[str] = Field(default_factory=list)

class MetricMapUpdate(BaseModel):
    term: str
    definition: str
    confidence: float = Field(ge=0, le=1)

class JoinMapUpdate(BaseModel):
    left: str
    right: str
    confidence: float = Field(ge=0, le=1)

class TableRoleUpdate(BaseModel):
    table: str
    role: Literal["fact", "dim", "bridge", "eav", "unknown"]
    confidence: float = Field(ge=0, le=1)

class SynonymUpdate(BaseModel):
    phrase: str
    maps_to: Dict[str, Any]
    confidence: float = Field(ge=0, le=1)

class SemanticUpsert(BaseModel):
    db_key: str
    metric_map_updates: List[MetricMapUpdate] = Field(default_factory=list)
    join_map_updates: List[JoinMapUpdate] = Field(default_factory=list)
    table_role_updates: List[TableRoleUpdate] = Field(default_factory=list)
    synonym_updates: List[SynonymUpdate] = Field(default_factory=list)
    notes: List[str] = Field(default_factory=list)

In [4]:
# ---------------------------
# ChromaDB RAG
# ---------------------------
import chromadb
from chromadb.utils import embedding_functions

CHROMA_DIR = os.getenv("CHROMA_DIR", "./chroma_ai_analytics")
chroma = chromadb.PersistentClient(path=CHROMA_DIR)

openai_key = os.getenv("OPENAI_API_KEY")
embedding_fn = None
if openai_key:
    embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=openai_key,
        model_name=os.getenv("CHROMA_OPENAI_EMBED_MODEL", "text-embedding-3-small"),
    )

def _safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_\-]", "_", s)[:60]

def get_collection(db_key: str):
    return chroma.get_or_create_collection(
        name=f"db_{_safe_name(db_key)}",
        embedding_function=embedding_fn,
        metadata={"db_key": db_key},
    )

def rag_retrieve(db_key: str, query: str, k: int = 8) -> str:
    col = get_collection(db_key)
    try:
        res = col.query(query_texts=[query], n_results=k)
        docs = res.get("documents", [[]])[0] or []
        return ("\n\n".join(docs[:k]))[:9000]
    except Exception:
        return ""

def rag_upsert_docs(db_key: str, items: List[Dict[str, Any]]):
    """
    items: [{"id": "...", "doc": "...", "meta": {...}}, ...]
    """
    col = get_collection(db_key)
    docs = [x["doc"] for x in items]
    ids = [x["id"] for x in items]
    metas = [x.get("meta", {}) for x in items]
    if docs:
        col.upsert(documents=docs, ids=ids, metadatas=metas)

def rag_upsert_catalog(db_key: str, catalog: DBCatalog):
    items = []
    # Table summaries
    for t, cols in catalog.table_columns.items():
        items.append({
            "id": f"table::{t}",
            "doc": (
                f"TABLE {t}\n"
                f"Columns: {', '.join([c.col for c in cols][:120])}\n"
                f"Candidate dates: {catalog.candidate_date_columns.get(t, [])}\n"
                f"Candidate measures: {catalog.candidate_measure_columns.get(t, [])}\n"
            ),
            "meta": {"type": "table", "table": t, "db_key": db_key},
        })
    # Fact candidates
    for ct in catalog.candidate_fact_tables:
        items.append({
            "id": f"fact::{ct.name}",
            "doc": f"FACT_CANDIDATE {ct.name}\nConfidence: {ct.confidence}\nWhy: {ct.why}",
            "meta": {"type": "fact_candidate", "table": ct.name, "db_key": db_key},
        })
    # EAV hints
    for e in catalog.eav_hints:
        items.append({
            "id": f"eav::{e.table}",
            "doc": (
                f"EAV_HINT {e.table}\nConfidence: {e.confidence}\nWhy: {e.why}\n"
                f"EntityID cols: {e.entity_id_cols}\nAttribute cols: {e.attribute_cols}\n"
                f"Value cols: {e.value_cols}\nTimestamp cols: {e.timestamp_cols}\n"
            ),
            "meta": {"type": "eav_hint", "table": e.table, "db_key": db_key},
        })
    # Joins
    for j in catalog.likely_joins:
        items.append({
            "id": f"join::{j.left}::{j.right}",
            "doc": f"JOIN {j.left} -> {j.right}\nConfidence: {j.confidence}\nWhy: {j.why}",
            "meta": {"type": "join", "db_key": db_key},
        })
    rag_upsert_docs(db_key, items)

def rag_upsert_semantics(db_key: str, upsert: SemanticUpsert):
    items = []
    for m in upsert.metric_map_updates:
        items.append({
            "id": f"metric::{m.term}",
            "doc": f"METRIC '{m.term}'\nDefinition: {m.definition}\nConfidence: {m.confidence}",
            "meta": {"type": "metric", "term": m.term, "db_key": db_key},
        })
    for j in upsert.join_map_updates:
        items.append({
            "id": f"trusted_join::{j.left}::{j.right}",
            "doc": f"TRUSTED_JOIN {j.left} -> {j.right}\nConfidence: {j.confidence}",
            "meta": {"type": "trusted_join", "db_key": db_key},
        })
    for s in upsert.synonym_updates:
        items.append({
            "id": f"syn::{s.phrase}",
            "doc": f"SYNONYM '{s.phrase}' maps_to={json.dumps(s.maps_to)}\nConfidence: {s.confidence}",
            "meta": {"type": "synonym", "phrase": s.phrase, "db_key": db_key},
        })
    rag_upsert_docs(db_key, items)

In [5]:
# ---------------------------
# JSON extraction + parsing
# ---------------------------
def extract_first_json(text: str) -> str:
    start = text.find("{")
    if start == -1:
        start = text.find("[")
    if start == -1:
        raise ValueError("No JSON found")
    stack = []
    for i in range(start, len(text)):
        ch = text[i]
        if ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            if not stack:
                continue
            open_ch = stack.pop()
            if (open_ch == "{" and ch != "}") or (open_ch == "[" and ch != "]"):
                raise ValueError("Mismatched braces")
            if not stack:
                return text[start:i+1]
    raise ValueError("Unclosed JSON")

def parse_model(model_cls, text: str):
    js = extract_first_json(text.strip())
    return model_cls.model_validate_json(js)

In [6]:
# ============================================================
# DETAILED AGENT INSTRUCTIONS (updated as requested)
# ============================================================

ORCH_INSTRUCTIONS = f"""
You are the Orchestrator for an AI analytics workflow.

Goal: deliver correct, executive-ready analysis quickly while delegating DB work to specialist agents.

Handoffs:
- Use Schema Scout to build/refresh a DB Catalog (esp. unknown DBs, EAV patterns).
- Use Query Runner (or DB Analyst in Step 1) to generate and execute complex SQL.
- Use Verifier to gate safety/correctness.
- Use Semantic Learner to update long-term memory (RAG) ONLY after PASS.
- Use Narrator to polish the final report if available.

Global constraints:
- Do NOT ask the user for permission to proceed. Work autonomously.
- Do NOT claim data is unavailable until discovery is complete AND at least one analysis query attempted.
- Prefer aggregates. Samples must be <=25 rows and avoid obvious PII-like columns.

Complex SQL:
- The system must support complex queries: multi-CTE, window functions, pivots, conditional aggregates, and EAV transformation.
- If EAV is detected (by Schema Scout or RAG), instruct Query Runner to use appropriate pivot/conditional aggregation strategies.

Charts and sources:
- Generate at most ONE chart if it clearly helps (time series, comparison, distribution).
- Use fetch only for general context (definitions/benchmarks) and NEVER include private data in fetch queries.

If RAG_CONTEXT is provided, treat it as hints about schema/joins/metrics learned previously.

Final output (unless Narrator exists):
- Executive Summary
- Key Insights
- Data Quality Notes
- Technical Notes
- SQL Appendix (exact SQL used)
- Sources (only if fetch used)
"""

DB_ANALYST_INSTRUCTIONS = f"""
You are DB Analyst. You perform BOTH discovery and analysis when used (Step 1 only).

Allowed tools: execute_query, list_tables, list_views, describe_table, sample_data, get_relationships.
Use execute_query for analysis and final outputs. list_tables is secondary; if it conflicts, trust INFORMATION_SCHEMA.

Mandatory workflow:
1) Discovery first (required):
   a) Query INFORMATION_SCHEMA.TABLES for BASE TABLE (and views if needed).
   b) Query INFORMATION_SCHEMA.COLUMNS for all columns.
   c) Search for relevant tables/columns using keywords from the user request (and synonyms).
2) Candidate selection:
   - Identify top 3–8 candidate tables.
   - Use describe_table for each.
   - Use minimal sample_data (<=10 rows) only when necessary to confirm semantics.
3) EAV detection:
   - Look for tables with patterns like (entity_id, attribute_id/name/key, value_* columns).
   - If EAV-like, identify how to pivot attributes into columns for analysis.
4) Analysis:
   - Use execute_query for all analysis queries.   
   - Use explicit ORDER BY for trends/rankings.
   - Avoid unordered TOP for inference.
   - Prefer aggregates; samples <=25 rows and avoid PII-like columns.
5) Missing data handling:
   - If ideal fields are missing, use best available proxies and clearly state assumptions/limitations.
6) Output:
Return STRICT JSON only matching:
{QueryRunResult.model_json_schema()}
- Put any schema/candidate notes into assumptions/caveats.
- sql_appendix must include the exact SQL used.
"""

SCHEMA_SCOUT_INSTRUCTIONS = f"""
You are Schema Scout. Your only job is to understand an unknown database schema quickly and safely,
and return a DB Catalog that enables complex analytics queries.

Allowed tools: execute_query (INFORMATION_SCHEMA ONLY), list_tables, describe_table, sample_data, get_relationships.
DO NOT run business analytics queries beyond lightweight inspection. Do NOT scan large tables.

Process (mandatory):
1) Primary discovery (SQL over INFORMATION_SCHEMA):
   - TABLES: get schema, table name, type (BASE TABLE / VIEW).
   - COLUMNS: get column names + types.
   - Identify candidates by keyword matching using the user request terms and general analytics terms:
     date/time, created/updated, status/state, amount/price/value, user/customer, id keys, etc.
   - If list_tables conflicts with INFORMATION_SCHEMA results, trust INFORMATION_SCHEMA.
2) Build table role hypotheses:
   - Fact-like tables: many columns, event/transaction-like names, have date/time + measure columns.
   - Dim-like tables: lookups (name/code/description), small, referenced by many FKs.
   - Bridge tables: two+ foreign keys, many-to-many.
3) Identify candidate date columns per table:
   - columns containing date/time patterns: created, updated, timestamp, date, datetime, time.
4) Identify candidate measure columns per table:
   - numeric/decimal/money patterns: amount, total, price, qty, count, duration, score, value.
5) Relationship discovery:
   - Use get_relationships where available.
   - Also infer possible joins from naming patterns: <table>Id, <entity>_id, id.
6) EAV detection (critical):
   - Identify tables that look like Entity-Attribute-Value:
     a) entity_id-like column(s)
     b) attribute_id/name/key column(s)
     c) one or more value columns (value, value_text, value_num, value_date, etc.)
   - For each EAV-like table, produce an EavHint describing entity/attribute/value/timestamp columns and why.
7) Minimal semantic validation:
   - For top 5–10 candidate tables only:
     - describe_table
     - sample_data <=10 rows (avoid obvious PII columns like email/phone/name/address when possible)
8) Output:
Return STRICT JSON only matching:
{DBCatalog.model_json_schema()}
Return ONLY JSON and nothing else.
"""

QUERY_RUNNER_INSTRUCTIONS = f"""
You are Query Runner. You receive:
- user request
- optional DB Catalog JSON (from Schema Scout)
- optional RAG_CONTEXT (learned schema/joins/metrics)

Your mission: answer the request using robust, potentially complex SQL.

Allowed tools: execute_query primarily. Use describe_table/get_relationships only if essential.

Mandatory workflow:
1) Interpret the request into a query plan:
   - target metric(s)
   - time window (if not specified, infer reasonable default and state it)
   - grain (daily/weekly/monthly)
   - segments (dimension breakdowns)
2) Choose tables/columns:
   - Prefer catalog-identified fact table and date column.
   - Use learned joins/metrics from RAG_CONTEXT when present.
3) Complex SQL allowed and encouraged when needed:
   - CTEs, window functions, conditional aggregates, subqueries.
   - If EAV detected:
     - Use conditional aggregation or PIVOT to materialize needed attributes.
     - Limit attribute selection to those relevant to the question.
     - If attribute names/ids are unclear, run ONE small discovery query to list top attributes by frequency.
4) Query discipline:
   - Total queries <=3 in fast mode.
   - Aggregates first; avoid large result sets.
   - Always ORDER BY for time series and rankings.
   - Never infer from unordered TOP.
5) Hybrid B constraints:
   - Avoid returning row-level data unless necessary.
   - If sampling is needed to interpret EAV, keep <=25 rows and exclude PII-like columns.
6) Output:
Return STRICT JSON only matching:
{QueryRunResult.model_json_schema()}
- answer_tables should be small but sufficient for charting (e.g., daily counts, top categories).
- sql_appendix must include exact SQL for each query.
"""

VERIFIER_INSTRUCTIONS = f"""
You are Verifier. You do not run tools. You act as a gatekeeper for safety, correctness, and Hybrid B compliance.

Input will include:
- user request
- sql_appendix (list of SQL strings)
- answer_tables (results)
- assumptions/caveats

Checks (mandatory):
1) Safety:
   - Queries must be SELECT-only.
   - Must not include INSERT/UPDATE/DELETE/MERGE/CREATE/ALTER/DROP/TRUNCATE.
   - Must not include EXEC, xp_cmdshell, sp_configure, OPENROWSET/OPENDATASOURCE.
   - No multi-statement batching with semicolons (one statement per query).
2) Correctness:
   - Time-series must ORDER BY date ascending.
   - Rankings must ORDER BY metric descending.
   - Date window and filters must match user request (e.g., “30 days ending on latest date”).
   - Avoid accidental duplication from joins (warn if likely).
3) Hybrid B:
   - Results should be aggregates; samples <=25 rows if present.
   - No obvious PII columns in outputs (email, phone, name, address).
4) Clarity:
   - Assumptions are explicit when schema is ambiguous or proxies were used.

Return STRICT JSON only matching:
{VerificationResult.model_json_schema()}
"""

SEMANTIC_LEARNER_INSTRUCTIONS = f"""
You are Semantic Learner. You do not run tools. You update long-term memory after a verified successful run.

Only proceed if verifier verdict == PASS.

Input:
- db_key
- user request
- DB Catalog (optional)
- sql_appendix
- answer_tables summary
- assumptions

Learn and emit UPSERT payload:
- metric_map_updates: map common business terms used by the user (e.g., revenue, completions) to SQL expressions used.
- join_map_updates: record joins that were used successfully (with confidence).
- table_role_updates: reinforce fact/dim/eav roles (with confidence).
- synonym_updates: map user phrases to tables/columns.

Return STRICT JSON only matching:
{SemanticUpsert.model_json_schema()}
"""

NARRATOR_INSTRUCTIONS = """
You are Narrator. You do not query databases.

Input:
- answer_tables
- assumptions/caveats
- sql_appendix
- optional chart outputs

Write an executive-ready report:
- Executive Summary (concise)
- Key Insights (bullets)
- Implications / Recommendations (optional)
- Data Quality Notes
- Technical Notes (short)
- SQL Appendix (exact SQL)
- Sources (only if fetch was used)
"""

In [7]:
# ---------------------------
# Build agents incrementally
# ---------------------------
def build_agents(step: int):
    helpers = {}

    db_analyst = Agent(
        name="DB Analyst",
        model="gpt-4o-mini",
        instructions=DB_ANALYST_INSTRUCTIONS,
        mcp_servers=[mssql_server],
    )
    schema_scout = Agent(
        name="Schema Scout",
        model="gpt-4o-mini",
        instructions=SCHEMA_SCOUT_INSTRUCTIONS,
        mcp_servers=[mssql_server],
    )
    query_runner = Agent(
        name="Query Runner",
        model="gpt-4o-mini",
        instructions=QUERY_RUNNER_INSTRUCTIONS,
        mcp_servers=[mssql_server],
    )
    verifier = Agent(
        name="Verifier",
        model="gpt-4o-mini",
        instructions=VERIFIER_INSTRUCTIONS,
        mcp_servers=[],
    )
    semantic_learner = Agent(
        name="Semantic Learner",
        model="gpt-4o-mini",
        instructions=SEMANTIC_LEARNER_INSTRUCTIONS,
        mcp_servers=[],
    )
    narrator = Agent(
        name="Narrator",
        model="gpt-4o-mini",
        instructions=NARRATOR_INSTRUCTIONS,
        mcp_servers=[chart_server, fetch_server],
    )

    helpers.update(
        db_analyst=db_analyst,
        schema_scout=schema_scout,
        query_runner=query_runner,
        verifier=verifier,
        semantic_learner=semantic_learner,
        narrator=narrator,
    )

    # Orchestrator has charts + fetch; no DB access
    orch_mcp = [chart_server, fetch_server]

    if step == 1:
        handoffs = [db_analyst]
    elif step == 2:
        handoffs = [schema_scout, query_runner]
    elif step == 3:
        handoffs = [schema_scout, query_runner, verifier]
    elif step == 4:
        handoffs = [schema_scout, query_runner, verifier, semantic_learner]
    else:
        handoffs = [schema_scout, query_runner, verifier, semantic_learner, narrator]

    orchestrator = Agent(
        name="Orchestrator",
        model="gpt-4o-mini",
        instructions=ORCH_INSTRUCTIONS,
        mcp_servers=orch_mcp,
        handoffs=handoffs,
    )

    return orchestrator, helpers

In [8]:
# ---------------------------
# End-to-end run (uses RAG context)
# ---------------------------
async def run_request(prompt: str, step: int, db_key: str):
    orchestrator, _ = build_agents(step)

    rag_ctx = rag_retrieve(db_key, prompt, k=8)
    prompt_with_rag = prompt + (f"\n\n[RAG_CONTEXT]\n{rag_ctx}\n" if rag_ctx else "")

    return await Runner.run(orchestrator, prompt_with_rag)

In [9]:
# ---------------------------
# Direct, testable building blocks
# ---------------------------
async def run_schema_scout(prompt: str, db_key: str) -> DBCatalog:
    _, helpers = build_agents(step=2)
    rag_ctx = rag_retrieve(db_key, prompt, k=8)
    p = prompt + (f"\n\n[RAG_CONTEXT]\n{rag_ctx}\n" if rag_ctx else "")
    res = await Runner.run(helpers["schema_scout"], p)
    catalog = parse_model(DBCatalog, res.final_output)
    rag_upsert_catalog(db_key, catalog)
    return catalog

async def run_query_runner(prompt: str, db_key: str) -> QueryRunResult:
    _, helpers = build_agents(step=2)
    rag_ctx = rag_retrieve(db_key, prompt, k=10)
    p = prompt + (f"\n\n[RAG_CONTEXT]\n{rag_ctx}\n" if rag_ctx else "")
    res = await Runner.run(helpers["query_runner"], p)
    return parse_model(QueryRunResult, res.final_output)

async def run_verifier(user_prompt: str, qres: QueryRunResult) -> VerificationResult:
    _, helpers = build_agents(step=3)
    payload = {
        "user_request": user_prompt,
        "sql_appendix": [q.model_dump() for q in qres.sql_appendix],
        "assumptions": qres.assumptions,
        "caveats": qres.caveats,
        "answer_tables": [t.model_dump() for t in qres.answer_tables],
    }
    res = await Runner.run(helpers["verifier"], json.dumps(payload))
    return parse_model(VerificationResult, res.final_output)

async def run_semantic_learner(db_key: str, user_prompt: str, catalog: Optional[DBCatalog], qres: QueryRunResult) -> SemanticUpsert:
    _, helpers = build_agents(step=4)
    payload = {
        "db_key": db_key,
        "user_request": user_prompt,
        "catalog": catalog.model_dump() if catalog else None,
        "sql_appendix": [q.model_dump() for q in qres.sql_appendix],
        "assumptions": qres.assumptions,
        "answer_tables": [t.model_dump() for t in qres.answer_tables],
        "verifier_verdict": "PASS",
    }
    res = await Runner.run(helpers["semantic_learner"], json.dumps(payload))
    upsert = parse_model(SemanticUpsert, res.final_output)
    rag_upsert_semantics(db_key, upsert)
    return upsert

In [ ]:
# ---------------------------
# Example usage
# ---------------------------
STEP = 5
DB_KEY = "mssql_demo"

USER_PROMPT = (
    "Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts "
    "for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. "
    "Only generate one chart."
)

async with mssql_server, fetch_server, chart_server:
    # 1) Scout once (good for messy/unknown DBs)
    catalog = await run_schema_scout(
        "Build a catalog for tables related to checklist completion, including dates and counts. Detect any EAV tables.",
        DB_KEY
    )

    # 2) Run query (complex SQL allowed)
    qres = await run_query_runner(USER_PROMPT, DB_KEY)

    # 3) Verify
    ver = await run_verifier(USER_PROMPT, qres)
    print("Verifier:", ver.verdict, ver.reasons[:2])

    # 4) Learn
    if ver.verdict == "PASS":
       _ = await run_semantic_learner(DB_KEY, USER_PROMPT, catalog, qres)

    # 5) Orchestrator end-to-end (uses RAG_CONTEXT)
    res = await run_request(USER_PROMPT, step=STEP, db_key=DB_KEY)
    display(Markdown(res.final_output))

```json
{
  "answer_tables": [
    {
      "name": "Daily Completion Counts for the Last 30 Days",
      "columns": [
        "CompletionDate",
        "CompletionCount"
      ],
      "rows": [
        ["2025-01-07", 26],
        ["2025-01-08", 15],
        ["2025-01-09", 18],
        ["2025-01-10", 15],
        ["2025-01-13", 105],
        ["2025-01-14", 26],
        ["2025-01-15", 19],
        ["2025-01-16", 23],
        ["2025-01-17", 25],
        ["2025-01-19", 2],
        ["2025-01-20", 80],
        ["2025-01-21", 36],
        ["2025-01-22", 14],
        ["2025-01-23", 15],
        ["2025-01-24", 6],
        ["2025-01-26", 1],
        ["2025-01-27", 101],
        ["2025-01-28", 18],
        ["2025-01-29", 14],
        ["2025-01-30", 18]
      ]
    }
  ],
  "assumptions": [
    "The latest CreationDate is accurately represented and used to determine the 30-day window for completion counts."
  ],
  "caveats": [
    "The dataset may have variability in completion counts due to external factors such as project deadlines or resource availability."
  ],
  "sql_appendix": [
    {
      "name": "Fetch Latest Creation Date",
      "sql": "SELECT MAX(CreationDate) AS LatestDate FROM dbo.CompletedChecklist;"
    },
    {
      "name": "Daily Completion Counts Query",
      "sql": "SELECT CAST(CreationDate AS DATE) AS CompletionDate, COUNT(*) AS CompletionCount \nFROM dbo.CompletedChecklist \nWHERE CreationDate >= DATEADD(DAY, -30, CAST((SELECT MAX(CreationDate) FROM dbo.CompletedChecklist) AS DATE)) \nGROUP BY CAST(CreationDate AS DATE) \nORDER BY CompletionDate;"
    }
  ]
}
```

In [11]:
catalog

DBCatalog(candidate_fact_tables=[CandidateTable(name='ChecklistAction', confidence=0.9, why='Contains action details with date columns and requires follow-ups.'), CandidateTable(name='CompletedChecklist', confidence=0.95, why='Tracks completed checklists with timestamps and user associations.')], table_columns={'Checklist': [ColumnInfo(col='ChecklistID', type='int'), ColumnInfo(col='ChecklistTemplateID', type='int'), ColumnInfo(col='ChecklistName', type='nvarchar(MAX)'), ColumnInfo(col='ChecklistDescription', type='nvarchar(MAX)'), ColumnInfo(col='CreationDate', type='datetime'), ColumnInfo(col='DeletionDate', type='datetime'), ColumnInfo(col='CreatedBy', type='int'), ColumnInfo(col='DeletedBy', type='int')], 'ChecklistAction': [ColumnInfo(col='ChecklistActionID', type='int'), ColumnInfo(col='ChecklistActionName', type='nvarchar(MAX)'), ColumnInfo(col='ChecklistActionDescription', type='nvarchar(MAX)'), ColumnInfo(col='CreationDate', type='datetime'), ColumnInfo(col='DueDate', type='in

In [13]:
qres

QueryRunResult(answer_tables=[AnswerTable(name='Daily Completion Counts for Completed Checklists', columns=['CompletionDate', 'CompletionCount'], rows=[['2025-01-07', 3], ['2025-01-08', 15], ['2025-01-09', 18], ['2025-01-10', 15], ['2025-01-13', 105], ['2025-01-14', 26], ['2025-01-15', 19], ['2025-01-16', 23], ['2025-01-17', 25], ['2025-01-19', 2], ['2025-01-20', 80], ['2025-01-21', 36], ['2025-01-22', 14], ['2025-01-23', 15], ['2025-01-24', 6], ['2025-01-26', 1], ['2025-01-27', 101], ['2025-01-28', 18], ['2025-01-29', 14], ['2025-01-30', 18]])], assumptions=['The latest CreationDate is considered as the endpoint for the 30-day range.', 'Completion counts reflect all checklist completions within the specified date range.'], caveats=['Counts may include multiple entries for the same checklist by different users.', 'Any filters or specific criteria on completion types are not applied.'], sql_appendix=[SQLQuery(name='Get Latest Creation Date', sql='SELECT MAX(CreationDate) AS LatestCreati

In [15]:
ver

VerificationResult(verdict='PASS', reasons=['All SQL queries are SELECT-only and do not include any disallowed operations.', 'Time-series data for daily completion counts is ordered by date ascending.', 'The 30-day window ends on the latest CreationDate as requested.', 'Results are aggregates and contain fewer than 25 rows.', 'No PII columns are present in the outputs.'], fixes=[])